In [11]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Suppress warnings
warnings.filterwarnings('ignore')

def load_and_preprocess_data(dataset_name, sample_size=10000):
    """
    Loads, cleans, and preprocesses a dataset while preserving IP addresses for blockchain storage.
    """
    print(f"\nLoading dataset: {dataset_name}")
    
    # Define dataset file paths
    dataset_paths = {
        "CICIDS2017": "archive/updated_tuesday.csv"
    }
    
    file_path = dataset_paths.get(dataset_name)
    if not file_path or not os.path.exists(file_path):
        print(f"Error: {dataset_name} dataset file not found. Check file path: {file_path}")
        return None
    
    data = pd.read_csv(file_path)
    
    # Standardize column names
    data.columns = data.columns.str.strip()
    
    # Rename label column if necessary
    label_mappings = {
        "CICIDS2017": "Label",
        "Bot-IoT": "label",
        "UNSW-NB15": "label"
    }
    
    label_column = label_mappings.get(dataset_name, "label")
    if label_column in data.columns:
        data.rename(columns={label_column: "intrusion_type"}, inplace=True)
    else:
        print(f"Error: '{label_column}' column not found in {dataset_name} dataset.")
        return None
    
    # Convert labels to binary classification
    data['intrusion_type'] = data['intrusion_type'].apply(lambda x: 'BENIGN' if x in ['BENIGN', 'Normal'] else 'ATTACK')
    
    # Replace inf/-inf with NaN
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Handle missing values
    imputer = SimpleImputer(strategy='mean')
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    data[numeric_features] = imputer.fit_transform(data[numeric_features])
    
    # Encode categorical features (excluding IPs)
    ip_columns = ['Source IP', 'Destination IP']
    for col in data.columns:
        if data[col].dtype == 'object' and col != "intrusion_type" and col not in ip_columns:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col])
    
    # Encode IP columns into new columns (keep originals as strings)
    for ip_col in ip_columns:
        if ip_col in data.columns:
            data[f'{ip_col} Encoded'] = LabelEncoder().fit_transform(data[ip_col])
    
    # Reduce dataset size if needed
    if sample_size and len(data) > sample_size:
        data = data.sample(n=sample_size, random_state=42)
    
    return data

def train_model(data, dataset_name):
    """
    Trains a model, identifies attack IPs, and prepares blockchain data.
    """
    if data is None:
        print(f"Error: {dataset_name} data not loaded. Check preprocessing steps.")
        return
    
    print(f"\nTraining model on {dataset_name}...")
    
    # Include ALL columns except 'intrusion_type' in X
    features = [col for col in data.columns if col != 'intrusion_type']
    X = data[features]
    y = data['intrusion_type']
    
    # Encode labels
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)
    
    # Split into train-test sets using ALL columns (IPs included)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Exclude original IPs from model training features
    model_features = [col for col in X.columns 
                     if col not in ['Source IP', 'Destination IP']]
    
    # Use model_features for training
    X_train_model = X_train[model_features]
    X_test_model = X_test[model_features]
    
    # Standardize feature values (only numerical features)
    numerical_features = [col for col in X_train_model.columns 
                         if X_train_model[col].dtype in ['int64', 'float64']]
    
    scaler = StandardScaler()
    X_train_model[numerical_features] = scaler.fit_transform(X_train_model[numerical_features])
    X_test_model[numerical_features] = scaler.transform(X_test_model[numerical_features])
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_model, y_train)
    
    # Predict on test data
    y_pred = model.predict(X_test_model)
    
    # Map predictions to original IPs (now available in X_test)
    prediction_results = pd.DataFrame({
        'Source IP': X_test['Source IP'],
        'Destination IP': X_test['Destination IP'],
        'Prediction': y_pred
    })
    
    # Convert predictions to labels
    prediction_results['Prediction'] = label_encoder.inverse_transform(y_pred)
    
    # Identify attack IPs
    attack_ips = prediction_results[prediction_results['Prediction'] == 'ATTACK']
    print(f"\nAttack IPs in {dataset_name}:")
    print(attack_ips[['Source IP', 'Destination IP']])
    
    # Save attack IPs for blockchain (optional)
    attack_ips.to_csv(f"attack_ips_{dataset_name}.csv", index=False)
    print(f"\nAttack IPs saved to: attack_ips_{dataset_name}.csv")
    
    # Evaluate model with NUMERICAL LABELS
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n{dataset_name} Accuracy: {accuracy:.4f}")
    print(f"{dataset_name} Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"{dataset_name} Classification Report:\n", classification_report(y_test, y_pred))

# Run for all three datasets
datasets = ["CICIDS2017",]
for dataset in datasets:
    data = load_and_preprocess_data(dataset)
    train_model(data, dataset)



Loading dataset: CICIDS2017

Training model on CICIDS2017...

Attack IPs in CICIDS2017:
              Source IP   Destination IP
339843     88.157.8.136    24.62.149.108
23045     33.180.83.133   107.105.102.32
52187      18.88.54.140    188.25.18.123
208396    29.235.95.216   184.55.163.135
11765    65.235.203.196    70.86.176.122
179304   217.64.232.101    27.210.151.25
20733    201.232.20.214      96.29.86.41
172221    87.160.64.222   122.67.168.109
361762  217.167.203.224   170.28.148.234
56269     101.75.191.74    89.157.145.15
22577   135.129.112.250   58.243.217.118
55022    215.125.38.133      99.66.15.43
65544    142.93.184.205   209.226.94.106
33771     37.170.248.23   34.232.255.221
45228    166.244.167.27  117.116.211.244
334258  133.183.133.244  210.187.197.154
206066       91.3.3.145  123.239.152.232
65717   109.144.240.253   191.202.234.35
28440   183.228.181.244      19.84.5.236
15357    177.37.178.117   62.162.178.121
289015   49.192.218.223    179.41.29.111
19060    